# Modelo genérico · Regressão Logística (com e sem regularização)

**Template reutilizável** para classificação **binária**: implementa do zero a sigmoide, a log loss, o gradiente e o gradiente descendente, com expansão polinomial opcional das features e regularização L2, e valida o resultado contra o `LogisticRegression` do scikit-learn.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 2. Aprendizado supervisionado › cartão *Regressão Logística* (`/modelar/aprendizado-supervisionado/#m-logistic`) |
| **Origem** | `C1_W3_Logistic_Regression.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; admissão por notas de duas provas; aprovação de microchips em dois testes), revisado, corrigido e generalizado |
| **Tipo de problema** | Supervisionado · classificação binária |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. Com `CAMINHO_DADOS = None`, roda com dados sintéticos de duas classes.
3. **Fronteira linear** (parte 2 do original): `GRAU_POLINOMIAL = 1`, `LAMBDA = 0`.
   **Fronteira não linear** (parte 3 do original): `GRAU_POLINOMIAL > 1` (o original usava 6, que gera 27 termos a partir de 2 features) e `LAMBDA > 0` para conter o sobreajuste.

### O que mudou em relação ao notebook original
- As duas partes do original (sem e com regularização) viraram **um único fluxo parametrizado** (`GRAU_POLINOMIAL`, `LAMBDA`), com custo e gradiente vetorizados; o `map_feature` do `utils.py` (fixo em 2 features e grau 6) foi trocado por `PolynomialFeatures`, que aceita qualquer número de features.
- `compute_cost` podia calcular `log(0)` quando a sigmoide satura: agora as probabilidades são limitadas a `[1e-15, 1 − 1e-15]` e o argumento da exponencial é recortado.
- O original avaliava só a **acurácia no próprio treino**. Entraram divisão treino/teste estratificada, padronização ajustada no treino e as métricas do cartão (ROC-AUC, PR-AUC, F1, log loss, Brier e matriz de confusão), além de uma tabela para escolher o limiar.
- `compute_gradient` perdeu a documentação no original; foi restaurada. `utils.py` e `public_tests.py` (ausentes) foram trocados por funções próprias e por uma verificação numérica do gradiente.
- Sem caminhos, colunas, rótulos ou valores do dataset original.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/clientes.csv"; None = dados sintéticos
SEPARADOR = None            # None = detecta; ex.: ";"
TEM_CABECALHO = True        # False se o arquivo não tem cabeçalho (como ex2data1.txt / ex2data2.txt)
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNA_ALVO = "alvo"        # coluna com duas classes
CLASSE_POSITIVA = None      # valor tratado como 1; None = usa 1 se o alvo já for 0/1
COLUNAS_FEATURES = None     # None = todas as demais colunas

# --- divisão e reprodutibilidade ---
TAMANHO_TESTE = 0.2
SEMENTE = 42

# --- modelo ---
GRAU_POLINOMIAL = 1         # 1 = fronteira linear; >1 = termos polinomiais e interações
LAMBDA = 0.0                # regularização L2 (lambda); 0 = sem regularização
TAXA_APRENDIZADO = 0.1      # alpha (features padronizadas)
NUM_ITERACOES = 5000
LIMIAR = 0.5                # probabilidade a partir da qual prevê a classe positiva

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, average_precision_score,
                             brier_score_loss, classification_report, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

np.random.seed(SEMENTE)

## 2. Carregamento dos dados

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n=300, semente=SEMENTE):
    """Duas classes em duas features, com sobreposição parcial."""
    rng = np.random.default_rng(semente)
    X = rng.normal(size=(n, 2)) * [12.0, 10.0] + [60.0, 65.0]
    logito = 0.12 * (X[:, 0] - 60) + 0.15 * (X[:, 1] - 65) + rng.logistic(scale=0.6, size=n)
    return pd.DataFrame({"feature_1": X[:, 0], "feature_2": X[:, 1], "alvo": (logito > 0).astype(int)})


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))
if COLUNA_ALVO not in df.columns:
    raise KeyError(f"COLUNA_ALVO={COLUNA_ALVO!r} não existe. Colunas: {list(df.columns)}")
df = df.dropna(subset=[COLUNA_ALVO])

classes = sorted(df[COLUNA_ALVO].unique().tolist())
if len(classes) != 2:
    raise ValueError(f"O alvo precisa ter exatamente 2 classes; encontrado: {classes}. "
                     "Para 3+ classes use softmax (template de rede neural multiclasse ou LogisticRegression).")
positiva = CLASSE_POSITIVA if CLASSE_POSITIVA is not None else (1 if set(classes) == {0, 1} else None)
if positiva is None:
    raise ValueError(f"Defina CLASSE_POSITIVA entre {classes}.")

features = COLUNAS_FEATURES or [c for c in df.columns if c != COLUNA_ALVO]
X_df = df[features]
y = (df[COLUNA_ALVO] == positiva).astype(float).to_numpy()
print("formato:", df.shape, "| classe positiva:", positiva)
print("proporção da classe positiva:", round(y.mean(), 3), "(muito longe de 0,5 = desbalanceado; veja a seção 7)")

In [ ]:
display(df.head())
faltantes = df.isna().sum()
print("faltantes por coluna:", faltantes[faltantes > 0].to_dict() or "nenhum")

In [ ]:
# Com duas features numéricas, o gráfico de dispersão do original (positivos × negativos).
num = X_df.select_dtypes(include="number").columns
if len(num) >= 2:
    a, b_ = num[0], num[1]
    for valor, marcador, rotulo in [(1, "+", f"{COLUNA_ALVO} = {positiva}"), (0, "o", "demais")]:
        sel = y == valor
        plt.scatter(X_df.loc[sel, a], X_df.loc[sel, b_], marker=marcador, alpha=0.7, label=rotulo)
    plt.xlabel(a); plt.ylabel(b_); plt.legend(); plt.title("Classes nas duas primeiras features numéricas")
    plt.show()

## 3. Preparação
Pré-processamento (one-hot sem a primeira categoria, para não haver colinearidade com o intercepto) → (opcional) expansão polinomial → padronização dos termos gerados. Tudo é ajustado **só no treino**; a divisão é estratificada para manter a proporção das classes.

In [ ]:
def montar_preprocessador(X, padronizar=True, remover_primeira=False):
    """Numéricas: mediana + padronização. Categóricas: moda + one-hot.
    remover_primeira=True descarta uma categoria por variável (evita a colinearidade perfeita
    com o intercepto em modelos lineares). Ajuste SÓ no treino (evita vazamento do teste)."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    passos_num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if padronizar else [])
    transformadores = []
    if numericas:
        transformadores.append(("num", make_pipeline(*passos_num), numericas))
    if categoricas:
        transformadores.append(("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          drop="first" if remover_primeira else None)), categoricas))
    return ColumnTransformer(transformadores, verbose_feature_names_out=False)


X_tr_df, X_te_df, y_train, y_test = train_test_split(
    X_df, y, test_size=TAMANHO_TESTE, random_state=SEMENTE, stratify=y)

etapas = [montar_preprocessador(X_tr_df, remover_primeira=True), StandardScaler()]   # padroniza também as dummies
if GRAU_POLINOMIAL > 1:
    etapas += [PolynomialFeatures(degree=GRAU_POLINOMIAL, include_bias=False), StandardScaler()]
transformador = make_pipeline(*etapas)
X_train = transformador.fit_transform(X_tr_df)
X_test = transformador.transform(X_te_df)
print(f"features originais: {X_df.shape[1]} → após preparação: {X_train.shape[1]}")
if X_train.shape[1] > 0.5 * len(X_train):
    print("ATENÇÃO: muitos termos para poucas linhas; aumente LAMBDA ou reduza GRAU_POLINOMIAL.")

## 4. Modelo

$$f_{\mathbf{w},b}(\mathbf{x}) = g(\mathbf{w}\cdot\mathbf{x} + b), \qquad g(z) = \frac{1}{1+e^{-z}}$$

Custo (log loss) com regularização L2 opcional — o viés $b$ não é regularizado:

$$J(\mathbf{w},b) = \frac{1}{m}\sum_{i=0}^{m-1}\left[-y^{(i)}\log f^{(i)} - (1-y^{(i)})\log(1-f^{(i)})\right] + \frac{\lambda}{2m}\sum_{j=0}^{n-1} w_j^2$$

Gradiente:

$$\frac{\partial J}{\partial b} = \frac{1}{m}\sum_i (f^{(i)} - y^{(i)}), \qquad \frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_i (f^{(i)} - y^{(i)})x_j^{(i)} + \frac{\lambda}{m}w_j$$

A fórmula é igual à da regressão linear, mas $f$ agora passa pela sigmoide. Com $\lambda = 0$ obtém-se a versão sem regularização.

In [ ]:
EPS = 1e-15


def sigmoid(z):
    """Sigmoide elemento a elemento; o recorte evita overflow em exp(-z)."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def compute_cost(X, y, w, b, lambda_=0.0):
    """Log loss média + termo L2 (lambda_/2m)·||w||². X: (m, n), y: (m,), w: (n,)."""
    m = X.shape[0]
    f = np.clip(sigmoid(X @ w + b), EPS, 1 - EPS)        # evita log(0)
    custo = -(y @ np.log(f) + (1 - y) @ np.log(1 - f)) / m
    return float(custo + lambda_ / (2 * m) * (w @ w))


def compute_gradient(X, y, w, b, lambda_=0.0):
    """Gradiente da log loss (regularizada). Devolve (dj_db, dj_dw), na mesma ordem do original."""
    m = X.shape[0]
    erro = sigmoid(X @ w + b) - y
    dj_db = float(erro.sum()) / m
    dj_dw = X.T @ erro / m + (lambda_ / m) * w
    return dj_db, dj_dw


def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters, lambda_, imprimir=10):
    """Gradiente descendente em lote; devolve w, b e o histórico do custo."""
    w, b = np.array(w_in, dtype=float).copy(), float(b_in)
    J_history = []
    passo = max(1, int(np.ceil(num_iters / imprimir)))
    for i in range(num_iters):
        dj_db, dj_dw = gradient_function(X, y, w, b, lambda_)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        J_history.append(cost_function(X, y, w, b, lambda_))
        if i % passo == 0 or i == num_iters - 1:
            print(f"Iteração {i:5d}: custo {J_history[-1]:8.4f}")
    return w, b, J_history


def predict_proba(X, w, b):
    return sigmoid(X @ w + b)


def predict(X, w, b, limiar=0.5):
    """1 se P(y = 1 | x) >= limiar, senão 0."""
    return (predict_proba(X, w, b) >= limiar).astype(int)

In [ ]:
# Checagens rápidas (substituem os testes do autograder).
assert sigmoid(0) == 0.5 and sigmoid(50) > 0.999 and sigmoid(-50) < 1e-3

rng = np.random.default_rng(0)
Xp, yp, wp, bp = rng.normal(size=(10, 4)), rng.integers(0, 2, 10).astype(float), rng.normal(size=4), 0.2
db, dw = compute_gradient(Xp, yp, wp, bp, lambda_=0.7)
h = 1e-6
dw_num = np.array([(compute_cost(Xp, yp, wp + h * e, bp, 0.7) - compute_cost(Xp, yp, wp - h * e, bp, 0.7)) / (2 * h)
                   for e in np.eye(4)])
db_num = (compute_cost(Xp, yp, wp, bp + h, 0.7) - compute_cost(Xp, yp, wp, bp - h, 0.7)) / (2 * h)
assert np.allclose(dw, dw_num, atol=1e-6) and np.isclose(db, db_num, atol=1e-6), "gradiente incorreto"
print("sigmoide e gradiente verificados")

## 5. Treinamento

In [ ]:
w0, b0 = np.zeros(X_train.shape[1]), 0.0
print(f"custo inicial: {compute_cost(X_train, y_train, w0, b0, LAMBDA):.4f} (log 2 = 0,693 com w = 0)")
w, b, J_hist = gradient_descent(X_train, y_train, w0, b0, compute_cost, compute_gradient,
                                TAXA_APRENDIZADO, NUM_ITERACOES, LAMBDA)

plt.figure(figsize=(6, 3.5))
plt.plot(J_hist)
plt.xlabel("iteração"); plt.ylabel("custo J"); plt.title("Convergência")
plt.show()

## 6. Avaliação
O original media a acurácia no treino. Aqui, as métricas do cartão no **teste**; a acurácia continua presente, para comparação.

In [ ]:
def avaliar(y_real, proba, limiar):
    prev = (proba >= limiar).astype(int)
    return {"acurácia": accuracy_score(y_real, prev), "precisão": precision_score(y_real, prev, zero_division=0),
            "recall": recall_score(y_real, prev), "F1": f1_score(y_real, prev),
            "ROC-AUC": roc_auc_score(y_real, proba), "PR-AUC": average_precision_score(y_real, proba),
            "log loss": log_loss(y_real, proba), "Brier": brier_score_loss(y_real, proba)}

p_train, p_test = predict_proba(X_train, w, b), predict_proba(X_test, w, b)
display(pd.DataFrame({"treino": avaliar(y_train, p_train, LIMIAR), "teste": avaliar(y_test, p_test, LIMIAR)}).T.round(4))
print(classification_report(y_test, (p_test >= LIMIAR).astype(int), digits=3, zero_division=0))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, (p_test >= LIMIAR).astype(int), ax=ax[0], colorbar=False)
ax[0].set_title(f"Matriz de confusão (limiar {LIMIAR})")
RocCurveDisplay.from_predictions(y_test, p_test, ax=ax[1])
ax[1].set_title("Curva ROC (teste)")
plt.tight_layout(); plt.show()

In [ ]:
# Escolha do limiar: 0,5 só é o melhor quando erros de cada tipo custam o mesmo.
# Escolha no treino/validação, nunca olhando o teste repetidamente.
limiares = np.round(np.arange(0.1, 0.91, 0.1), 2)
tabela = pd.DataFrame([{"limiar": t, "precisão": precision_score(y_train, p_train >= t, zero_division=0),
                        "recall": recall_score(y_train, p_train >= t), "F1": f1_score(y_train, p_train >= t)}
                       for t in limiares]).set_index("limiar")
display(tabela.round(3))

In [ ]:
# Fronteira de decisão (quando há exatamente duas features numéricas de entrada), como no original.
if X_df.shape[1] == 2 and X_df.select_dtypes(include="number").shape[1] == 2:
    a, c = X_df.columns
    xa = np.linspace(X_df[a].min(), X_df[a].max(), 200)
    xc = np.linspace(X_df[c].min(), X_df[c].max(), 200)
    A, C = np.meshgrid(xa, xc)
    grade = pd.DataFrame({a: A.ravel(), c: C.ravel()})
    P = predict_proba(transformador.transform(grade), w, b).reshape(A.shape)
    plt.contour(A, C, P, levels=[LIMIAR], linewidths=2)
    for valor, marcador in [(1, "+"), (0, "o")]:
        sel = y == valor
        plt.scatter(X_df.loc[sel, a], X_df.loc[sel, c], marker=marcador, alpha=0.6, label=f"y = {valor}")
    plt.xlabel(a); plt.ylabel(c); plt.legend(); plt.title(f"Fronteira de decisão (grau {GRAU_POLINOMIAL}, λ = {LAMBDA})")
    plt.show()
else:
    print("Fronteira só é desenhada com duas features numéricas de entrada.")

## 7. Validação com a referência (scikit-learn)
O `LogisticRegression` minimiza $C\sum_i \text{loss}_i + \tfrac12\lVert\mathbf{w}\rVert^2$, que equivale ao custo acima com $C = 1/\lambda$ ($\lambda = 0$ → `C=np.inf`, sem penalização). Coeficientes próximos confirmam a implementação; diferenças pequenas vêm de o gradiente descendente ainda não ter convergido totalmente.

Para dados desbalanceados, o cartão recomenda `class_weight="balanced"` na versão de produção.

In [ ]:
C = np.inf if LAMBDA == 0 else 1.0 / LAMBDA
ref = LogisticRegression(C=C, max_iter=10_000).fit(X_train, y_train)
p_ref = ref.predict_proba(X_test)[:, 1]
dif = np.max(np.abs(np.r_[w, b] - np.r_[ref.coef_.ravel(), ref.intercept_]))
print(f"maior diferença entre coeficientes: {dif:.4f}")
if dif > 0.1:
    print("Os coeficientes ainda diferem: o gradiente descendente não convergiu por completo (comum com muitos "
          "termos polinomiais e LAMBDA pequeno). Aumente NUM_ITERACOES; compare sobretudo as previsões abaixo.")
print(f"ROC-AUC teste · do zero: {roc_auc_score(y_test, p_test):.4f} | scikit-learn: {roc_auc_score(y_test, p_ref):.4f}")
print(f"concordância das previsões: {np.mean((p_test >= LIMIAR) == (p_ref >= LIMIAR)):.1%}")

## 8. Previsão para novos dados

In [ ]:
novos = X_df.sample(3, random_state=SEMENTE)      # >>> ADAPTAR: DataFrame com as mesmas colunas de `features`
proba = predict_proba(transformador.transform(novos), w, b)
outra = next(c for c in classes if c != positiva)
display(novos.assign(probabilidade=proba.round(4), previsao=[positiva if p >= LIMIAR else outra for p in proba]))

## 9. Próximos passos (segundo o cartão do sistema)
- Pressuposto: log-odds lineares nas features (ou nos termos polinomiais gerados). Regra prática de volume: ~10 eventos da classe minoritária por feature.
- `exp(w_j)` é a razão de chances da feature *j* (por desvio-padrão, já que as features foram padronizadas).
- **Prefira boosting** quando houver interações relevantes e volume suficiente; **Naive Bayes** em texto com pouquíssimos dados.